3.1. Wrangling Data with MongoDB 🇰🇪


In [1]:
from pprint import PrettyPrinter

import pandas as pd
from pymongo import MongoClient


Next, we Instantiate a `PrettyPrinter`, and assign it to the variable `pp`.


In [4]:
pp = PrettyPrinter (indent = 2)

# Prepare Data


## Connect


🛠️ Instruction: Locate the IP address of the machine running MongoDB and assign it to the variable host. Make sure to use a string (i.e., wrap the IP in quotes).

⚠️ Note: The IP address is dynamic — it may change every time you start the lab. Always check the current IP before proceeding.


In [5]:
host = "192.97.115.2"

Next, we Create a client that connects to the database running at `host` on port `27017`.


In [7]:
client = MongoClient(host="192.97.115.2",port=27017)

## Explore


Next, we Print a list of the databases available on `client`.


In [9]:
from sys import getsizeof
my_list = [0,1,2,3,4,5,6,7,8]
my_range = range(5,9)
for i in my_range:
    print(i)

5
6
7
8


In [10]:
getsizeof(my_list)

136

In [11]:
pp.pprint(list(client.list_databases()))

[ {'empty': False, 'name': 'admin', 'sizeOnDisk': 40960},
  {'empty': False, 'name': 'air-quality', 'sizeOnDisk': 5931008},
  {'empty': False, 'name': 'config', 'sizeOnDisk': 61440},
  {'empty': False, 'name': 'local', 'sizeOnDisk': 73728},
  {'empty': False, 'name': 'wqu-abtest', 'sizeOnDisk': 585728}]


Next, we Assign the `"air-quality"` database to the variable `db`.


In [13]:
db = client["air-quality"]

Next, we Use the [`list_collections`](https://pymongo.readthedocs.io/en/stable/api/pymongo/database.html?highlight=list_collections#pymongo.database.Database.list_collections) method to print a list of the collections available in `db`.


In [15]:
for c in db.list_collections():
    print(c["name"])

dar-es-salaam
system.buckets.dar-es-salaam
nairobi
system.buckets.nairobi
system.views
lagos
system.buckets.lagos


Next, we Assign the `"nairobi"` collection in `db` to the variable name `nairobi`.


In [18]:
nairobi = db["nairobi"]

Next, we Use the [`count_documents`](https://pymongo.readthedocs.io/en/stable/api/pymongo/collection.html#pymongo.collection.Collection.count_documents) method to see how many documents are in the `nairobi` collection.


In [21]:
nairobi.count_documents({})

202212

Next, we Use the [`find_one`](https://pymongo.readthedocs.io/en/stable/api/pymongo/collection.html#pymongo.collection.Collection.find_one) method to retrieve one document from the `nairobi` collection, and assign it to the variable name `result`.


In [23]:
result = nairobi.find_one({})
pp.pprint(result)

{ '_id': ObjectId('6525d772f44bfedd842a6fcc'),
  'metadata': { 'lat': -1.3,
                'lon': 36.785,
                'measurement': 'temperature',
                'sensor_id': 58,
                'sensor_type': 'DHT22',
                'site': 29},
  'temperature': 16.5,
  'timestamp': datetime.datetime(2018, 9, 1, 0, 0, 4, 301000)}


Next, we Use the [`distinct`](https://pymongo.readthedocs.io/en/stable/api/pymongo/collection.html#pymongo.collection.Collection.distinct) method to determine how many sensor sites are included in the `nairobi` collection.


In [25]:
nairobi.distinct("metadata.site")

[6, 29]

Next, we Use the [`count_documents`](https://pymongo.readthedocs.io/en/stable/api/pymongo/collection.html#pymongo.collection.Collection.count_documents) method to determine how many readings there are for each site in the `nairobi` collection.


In [29]:

print("Documents from site 6:", nairobi.count_documents({"metadata.site":6}))
print("Documents from site 29:",nairobi.count_documents({"metadata.site":29}) )

Documents from site 6: 70360
Documents from site 29: 131852


Next, we Use the [`aggregate`](https://pymongo.readthedocs.io/en/stable/api/pymongo/collection.html#pymongo.collection.Collection.aggregate) method to determine how many readings there are for each site in the `nairobi` collection.


In [35]:
result = nairobi.aggregate(
    [
        {"$group": {"_id": "$metadata.site", "count": {"$count":{}}}}
    ]

)
pp.pprint(list(result))

[{'_id': 6, 'count': 70360}, {'_id': 29, 'count': 131852}]


Next, we Use the [`distinct`](https://pymongo.readthedocs.io/en/stable/api/pymongo/collection.html#pymongo.collection.Collection.distinct) method to determine how many types of measurements have been taken in the `nairobi` collection.


In [37]:
nairobi.distinct("metadata.measurement")

['P2', 'temperature', 'humidity', 'P1']

Next, we Use the [`find`](https://pymongo.readthedocs.io/en/stable/api/pymongo/collection.html#pymongo.collection.Collection.find) method to retrieve the PM 2.5 readings from all sites. Be sure to limit your results to 3 records only.


In [42]:
result = nairobi.find({"metadata.measurement": "P2"}).limit(3)
pp.pprint(list(result))

[ { 'P2': 34.43,
    '_id': ObjectId('6525d775f44bfedd842bf24d'),
    'metadata': { 'lat': -1.3,
                  'lon': 36.785,
                  'measurement': 'P2',
                  'sensor_id': 57,
                  'sensor_type': 'SDS011',
                  'site': 29},
    'timestamp': datetime.datetime(2018, 9, 1, 0, 0, 2, 472000)},
  { 'P2': 30.53,
    '_id': ObjectId('6525d775f44bfedd842bf24e'),
    'metadata': { 'lat': -1.3,
                  'lon': 36.785,
                  'measurement': 'P2',
                  'sensor_id': 57,
                  'sensor_type': 'SDS011',
                  'site': 29},
    'timestamp': datetime.datetime(2018, 9, 1, 0, 5, 3, 941000)},
  { 'P2': 22.8,
    '_id': ObjectId('6525d775f44bfedd842bf24f'),
    'metadata': { 'lat': -1.3,
                  'lon': 36.785,
                  'measurement': 'P2',
                  'sensor_id': 57,
                  'sensor_type': 'SDS011',
                  'site': 29},
    'timestamp': datetime.datetime(

Next, we Use the [`aggregate`](https://pymongo.readthedocs.io/en/stable/api/pymongo/collection.html#pymongo.collection.Collection.aggregate) method to calculate how many readings there are for each type (`"humidity"`, `"temperature"`, `"P2"`, and `"P1"`) in site `6`.


In [46]:
result = nairobi.aggregate(
    [
        {"$match": {"metadata.site": 6}},
        {"$group": {"_id": "$metadata.measurement", "count": {"$count":{}}}}
    ]

)
pp.pprint(list(result))

[ {'_id': 'P2', 'count': 18169},
  {'_id': 'temperature', 'count': 17011},
  {'_id': 'humidity', 'count': 17011},
  {'_id': 'P1', 'count': 18169}]


Next, we Use the [`aggregate`](https://pymongo.readthedocs.io/en/stable/api/pymongo/collection.html#pymongo.collection.Collection.aggregate) method to calculate how many readings there are for each type (`"humidity"`, `"temperature"`, `"P2"`, and `"P1"`) in site `29`.


In [48]:
result = nairobi.aggregate(
    [
        {"$match": {"metadata.site": 29}},
        {"$group": {"_id": "$metadata.measurement", "count": {"$count":{}}}}
    ]

)
pp.pprint(list(result))

[ {'_id': 'P2', 'count': 32907},
  {'_id': 'temperature', 'count': 33019},
  {'_id': 'humidity', 'count': 33019},
  {'_id': 'P1', 'count': 32907}]


## Import


Next, we Use the [`find`](https://pymongo.readthedocs.io/en/stable/api/pymongo/collection.html#pymongo.collection.Collection.find) method to retrieve the PM 2.5 readings from site `29`. Be sure to limit your results to 3 records only. Since we won't need the metadata for our model, use the `projection` argument to limit the results to the `"P2"` and `"timestamp"` keys only.


In [55]:
result = nairobi.find(
    {"metadata.site": 29, "metadata.measurement": "P2"},
    projection={"P2": 1, "timestamp": 1, "_id": 0}
)
#pp.pprint(result.next())

Next, we Read records from your `result` into the DataFrame `df`. Be sure to set the index to `"timestamp"`.


In [56]:
df = pd.DataFrame(result).set_index("timestamp")
df.head()

,P2
timestamp,
2018-09-01 00:00:02.472,34.43
2018-09-01 00:05:03.941,30.53
2018-09-01 00:10:04.374,22.80
2018-09-01 00:15:04.245,13.30
2018-09-01 00:20:04.869,16.57


In [57]:
# Check your work
assert df.shape[1] == 1, f"`df` should have only one column, not {df.shape[1]}."
assert df.columns == [
    "P2"
], f"The single column in `df` should be `'P2'`, not {df.columns[0]}."
assert isinstance(df.index, pd.DatetimeIndex), "`df` should have a `DatetimeIndex`."

---

content is licensed solely for personal use. Redistribution or
publication of this material is strictly prohibited.
